In [1]:
from transformers import (VisionEncoderDecoderModel,
                          ViTModel,GPT2LMHeadModel,
                          AutoTokenizer,ViTImageProcessor,
                          Trainer,TrainingArguments)
from typing import List, Any 
import torch
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from datasets import load_dataset,Dataset
import os
from tqdm import tqdm 
import numpy as np 
import pandas as pd 
from transformers.integrations import TensorBoardCallback
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import codecs
import os
import csv
import torch
from PIL import Image
from transformers import AutoTokenizer, ViTImageProcessor, VisionEncoderDecoderModel
from nltk.translate.bleu_score import sentence_bleu
from tqdm import tqdm
import nltk
import jieba
from nltk.translate.meteor_score import meteor_score
from rouge import Rouge
exp_names = ['a_geovit_geogpt', 
             'a_16384_geogpt',
             'a_32384_geogpt',
             'a_large16224_geogpt',
             'a_16224_geogpt', 
             'a_geovit_geogpt_40', 
             'a_geovit_geogpt_80',
             'a_geovit_gpt_wechsel',
             'a_geovit_gpt_chinese']
VIT_MODEL_NAME_OR_PATHs = ['D:/code/geovit/models/checkpoint-7500', 
                           'google/vit-base-patch16-384',
                           'google/vit-base-patch32-384', 
                           'google/vit-large-patch16-224',
                           'google/vit-base-patch16-224',
                           'D:/code/geovit/models/checkpoint-7500', 
                           'D:/code/geovit/models/checkpoint-7500', 
                           'D:/code/geovit/models/checkpoint-7500',
                           'D:/code/geovit/models/checkpoint-7500']
GPT_MODEL_NAME_OR_PATHs = ['D:/code/geogpt/gpt2_geo/checkpoint-128000',
                           'D:/code/geogpt/gpt2_geo/checkpoint-128000',
                           'D:/code/geogpt/gpt2_geo/checkpoint-128000', 
                           'D:/code/geogpt/gpt2_geo/checkpoint-128000',
                           'D:/code/geogpt/gpt2_geo/checkpoint-128000', 
                           'D:/code/geogpt/gpt2_geo_40/checkpoint-32000',
                           'D:/code/geogpt/gpt2_geo_80/checkpoint-84000',
                           'benjamin/gpt2-wechsel-chinese',
                           'yuanzhoulvpi/gpt2_chinese']
for exp_name, VIT_MODEL_NAME_OR_PATH, GPT_MODEL_NAME_OR_PATH in zip(exp_names, VIT_MODEL_NAME_OR_PATHs, GPT_MODEL_NAME_OR_PATHs):


    BLANK_FILE_PATH = '196blank.csv'
    picnum = 392
    BLANK_AVE_FILE_PATH = "10blank_grs.csv"
    zidian = 'glossary.csv'
    grs = 0
    csv_name = 'geo2k'
    os.mkdir(exp_name)

    data = pd.read_csv(csv_name + '.csv')

    # 按指定比例拆分数据集
    train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

    # 修改文件名
    train_filename = csv_name + '_train.csv'
    test_filename = csv_name + '_test.csv'

    # 将拆分后的数据保存为新的CSV文件，带BOM的UTF-8编码
    with tqdm(total=2) as pbar:
        train_data.to_csv(codecs.open(os.path.join(exp_name, train_filename), 'w', 'utf-8-sig'), index=False)
        pbar.update(1)
        test_data.to_csv(codecs.open(os.path.join(exp_name, test_filename), 'w', 'utf-8-sig'), index=False)
        pbar.update(1)


    #GPT_MODEL_NAME_OR_PATH = "yuanzhoulvpi/gpt2_chinese"
    #vision_encoder_decoder_model_name_or_path = "yuanzhoulvpi/vit-gpt2-image-chinese-captioning"


    VIT_model = ViTModel.from_pretrained(VIT_MODEL_NAME_OR_PATH)
    GPT_model = GPT2LMHeadModel.from_pretrained(GPT_MODEL_NAME_OR_PATH, add_cross_attention=True)
    GPT_model.config.add_cross_attention# = True
    processor = ViTImageProcessor.from_pretrained(VIT_MODEL_NAME_OR_PATH)
    tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_NAME_OR_PATH)
    if exp_name == 'a_geovit_gpt_wechsel':
        tokenizer.pad_token = tokenizer.eos_token    
    def process_image_2_pixel_value(x:str) -> Tensor:
        image = Image.open(x)
        res = processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        return res 
    def process_text_2_input_id(x:str) :
        res = tokenizer(text=x,max_length=100, truncation=True,padding="max_length")['input_ids']
        return res 
    tokenizer.pad_token_id
    new_encoder_decoder_model = VisionEncoderDecoderModel(
        encoder=VIT_model,
        decoder=GPT_model,

    )
    new_encoder_decoder_model.config.decoder_start_token_id = tokenizer.bos_token_id
    new_encoder_decoder_model.config.pad_token_id = tokenizer.pad_token_id
    torch.tensor(process_text_2_input_id(x='hhh'), dtype=torch.long).unsqueeze(0).shape
    new_encoder_decoder_model.config.add_cross_attention = True
    new_encoder_decoder_model.config.add_cross_attention
    dataset = Dataset.from_pandas(df=pd.read_csv(os.path.join(exp_name, train_filename)))
    dataset = dataset.train_test_split(test_size=0.25)
    def tokenizer_text(examples) :
        examples['labels'] = [process_text_2_input_id(i) for i in examples['text']]
        # res = [process_text_2_input_id(i) for i in examples['text']]
        # examples['labels'] = [i['input_ids'] for i in res]
        return examples
    def transform_images(examples):
        images = [process_image_2_pixel_value(i) for i in examples['image_path']]
        # images = [torch.Tensor(i) for i in images]
        examples['pixel_values'] = images
        return examples
    dataset = dataset.map(
        function=tokenizer_text,
        batched=True
    )
    dataset.set_transform(transform=transform_images)
    def collate_fn(examples):
        pixel_values = torch.stack([i['pixel_values'] for i in examples])
        labels = torch.tensor([example["labels"] for example in examples], dtype=torch.long)
        return {
            "pixel_values": pixel_values,
            "labels": labels
        }
    train_argument = TrainingArguments(
        output_dir=exp_name,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        evaluation_strategy="steps",
        eval_steps=200,
        logging_strategy='epoch',
        logging_dir='logs_' + exp_name,
        gradient_accumulation_steps=8,
        num_train_epochs=10,
        weight_decay=0.1,
        lr_scheduler_type="cosine",
        learning_rate=1e-5,
        save_strategy="epoch",  # 按epoch保存模型
        fp16=True,
        remove_unused_columns=False,
        save_total_limit=10,
    )
    trainer = Trainer(
        model=new_encoder_decoder_model,
        args=train_argument,
        train_dataset=dataset['train'],
        eval_dataset=dataset['test'],
        data_collator=collate_fn,
        callbacks=[TensorBoardCallback()]
    )
    trainer.train()

    print(exp_name)
    print("训练完成，开始评价")

    models_root = 'D:/code/vit-gpt2/'+ exp_name

    processor = ViTImageProcessor.from_pretrained(VIT_MODEL_NAME_OR_PATH)
    tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_NAME_OR_PATH)

    e = 0

    contents = os.listdir(models_root)
    folders = [item for item in contents if os.path.isdir(os.path.join(models_root, item))]
    folders.sort(key= lambda x:int(x[11:]))

    with open(BLANK_AVE_FILE_PATH, "r", encoding="utf-8") as avefile:

        avecsv_reader = csv.reader(avefile)
        averows = list(avecsv_reader)

        for model_name in folders:
            e = e + 1
            vision_encoder_decoder_model_name_or_path = os.path.join(models_root, model_name)
            model = VisionEncoderDecoderModel.from_pretrained(vision_encoder_decoder_model_name_or_path)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            max_length = 20
            num_beams = 4
            gen_kwargs = {"max_length": max_length, "num_beams": num_beams}

            def predict(image_paths):
                images = []
                for image_path in image_paths:
                    i_image = Image.open(image_path)
                    if i_image.mode != "RGB":
                        i_image = i_image.convert(mode="RGB")
                    images.append(i_image)

                pixel_values = processor(images=images, return_tensors="pt").pixel_values
                pixel_values = pixel_values.to(device)

                output_ids = model.generate(pixel_values, **gen_kwargs)

                preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
                preds = [pred.strip() for pred in preds]
                return preds

            # 读取BBLANK_FILE_PATH文件
            with open(BLANK_FILE_PATH, "r", encoding="utf-8") as file:
                csv_reader = csv.reader(file)
                rows = list(csv_reader)

            # 补全第三列infer的文字

            sumbleu1 = 0
            sumbleu2 = 0
            sumbleu2s = 0
            sumbleu3 = 0
            sumbleu3s = 0
            sumbleu4 = 0
            sumbleu4s = 0
            sumMETEOR = 0
            sumROUGE1r = 0
            sumROUGE1p = 0
            sumROUGE1f = 0
            sumROUGE2r = 0
            sumROUGE2p = 0
            sumROUGE2f = 0
            sumROUGELr = 0
            sumROUGELp = 0
            sumROUGELf = 0
            sumjiebableu1 = 0
            sumjiebableu2 = 0
            sumjiebableu2s = 0
            sumjiebableu3 = 0
            sumjiebableu3s = 0
            sumjiebableu4 = 0
            sumjiebableu4s = 0
            sumjiebaMETEOR = 0
            sumjiebaROUGE1r = 0
            sumjiebaROUGE1p = 0
            sumjiebaROUGE1f = 0
            sumjiebaROUGE2r = 0
            sumjiebaROUGE2p = 0
            sumjiebaROUGE2f = 0
            sumjiebaROUGELr = 0
            sumjiebaROUGELp = 0
            sumjiebaROUGELf = 0
            sumgrs = 0
    #        unique_strings = set()  # 创建一个空的集合
            for i, row in tqdm(enumerate(rows), total=len(rows)-1):
                if i == 0:
                    continue
                image_path = row[0]
                reference = row[1]
                infer = predict([image_path])[0]
                infer = infer.replace(' ','')
                rows[i][2] = infer
     #           unique_strings.add(infer)
                # 计算BLEU4得分
                reference_tokens_jieba = jieba.lcut(reference)
                rows[i][3] = reference_tokens_jieba
                infer_tokens_jieba = jieba.lcut(infer)
                rows[i][4] = infer_tokens_jieba

                reference_tokens = list(reference)
                rows[i][5] =  reference_tokens
                infer_tokens = list(infer)
                rows[i][6] = infer_tokens

                rows[i][7] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1., 0, 0, 0))
                sumbleu1 = sumbleu1 + rows[i][7]

                rows[i][8] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./2., 1./2.))
                sumbleu2 = sumbleu2 + rows[i][8]

                rows[i][9] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./2., 1./2.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumbleu2s = sumbleu2s + rows[i][9]

                rows[i][10] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./3., 1./3., 1./3.))
                sumbleu3 = sumbleu3 + rows[i][10]

                rows[i][11] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./3., 1./3., 1./3.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumbleu3s = sumbleu3s + rows[i][11]

                rows[i][12] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./4., 1./4., 1./4., 1./4.))
                sumbleu4 = sumbleu4 + rows[i][12]

                rows[i][13] = nltk.translate.bleu_score.sentence_bleu([reference_tokens], infer_tokens, weights=(1./4., 1./4., 1./4., 1./4.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumbleu4s = sumbleu4s + rows[i][13]

                rows[i][14] = nltk.translate.meteor_score.meteor_score([reference_tokens], infer_tokens)
                sumMETEOR =  sumMETEOR + rows[i][14] 


                rouge=Rouge()
                if len(infer) <= 0:
                    infer = '无法生成'
                rough_infer = ' '.join(list(infer))
                rough_reference = ' '.join(list(reference))
                roguescores = rouge.get_scores(rough_infer,rough_reference)  
                rows[i][15] = roguescores[0]['rouge-1']['r']
                rows[i][16] = roguescores[0]['rouge-1']['p']
                rows[i][17] = roguescores[0]['rouge-1']['f']
                rows[i][18] = roguescores[0]['rouge-2']['r']
                rows[i][19] = roguescores[0]['rouge-2']['p']
                rows[i][20] = roguescores[0]['rouge-2']['f']
                rows[i][21] = roguescores[0]['rouge-l']['r']
                rows[i][22] = roguescores[0]['rouge-l']['p']
                rows[i][23] = roguescores[0]['rouge-l']['f']
                sumROUGE1r =  sumROUGE1r + rows[i][15]
                sumROUGE1p =  sumROUGE1p + rows[i][16]
                sumROUGE1f =  sumROUGE1f + rows[i][17]
                sumROUGE2r =  sumROUGE2r + rows[i][18]
                sumROUGE2p =  sumROUGE2p + rows[i][19]
                sumROUGE2f =  sumROUGE2f + rows[i][20]
                sumROUGELr =  sumROUGELr + rows[i][21]
                sumROUGELp =  sumROUGELp + rows[i][22]
                sumROUGELf =  sumROUGELf + rows[i][23]


                rows[i][24] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1., 0, 0, 0))
                sumjiebableu1 = sumjiebableu1 + rows[i][24]

                rows[i][25] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./2., 1./2.))
                sumjiebableu2 = sumjiebableu2 + rows[i][25]

                rows[i][26] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./2., 1./2.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumjiebableu2s = sumjiebableu2s + rows[i][26]

                rows[i][27] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./3., 1./3., 1./3.))
                sumjiebableu3 = sumjiebableu3 + rows[i][27]

                rows[i][28] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./3., 1./3., 1./3.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumjiebableu3s = sumjiebableu3s + rows[i][28]

                rows[i][29] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./4., 1./4., 1./4., 1./4.))
                sumjiebableu4 = sumjiebableu4 + rows[i][29]

                rows[i][30] = nltk.translate.bleu_score.sentence_bleu([reference_tokens_jieba], infer_tokens_jieba, weights=(1./4., 1./4., 1./4., 1./4.), smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method2)
                sumjiebableu4s = sumjiebableu4s + rows[i][30]

                rows[i][31] = nltk.translate.meteor_score.meteor_score([reference_tokens_jieba], infer_tokens_jieba)
                sumjiebaMETEOR =  sumjiebaMETEOR + rows[i][31] 

                rouge=Rouge()
                if len(infer) <= 0:
                    infer = '无法生成'
                rough_infer = ' '.join(jieba.cut(infer))
                rough_reference = ' '.join(jieba.cut(reference))
                roguescores = rouge.get_scores(rough_infer,rough_reference)  
                rows[i][32] = roguescores[0]['rouge-1']['r']
                rows[i][33] = roguescores[0]['rouge-1']['p']
                rows[i][34] = roguescores[0]['rouge-1']['f']
                rows[i][35] = roguescores[0]['rouge-2']['r']
                rows[i][36] = roguescores[0]['rouge-2']['p']
                rows[i][37] = roguescores[0]['rouge-2']['f']
                rows[i][38] = roguescores[0]['rouge-l']['r']
                rows[i][39] = roguescores[0]['rouge-l']['p']
                rows[i][40] = roguescores[0]['rouge-l']['f']
                sumjiebaROUGE1r =  sumjiebaROUGE1r + rows[i][32]
                sumjiebaROUGE1p =  sumjiebaROUGE1p + rows[i][33]
                sumjiebaROUGE1f =  sumjiebaROUGE1f + rows[i][34]
                sumjiebaROUGE2r =  sumjiebaROUGE2r + rows[i][35]
                sumjiebaROUGE2p =  sumjiebaROUGE2p + rows[i][36]
                sumjiebaROUGE2f =  sumjiebaROUGE2f + rows[i][37]
                sumjiebaROUGELr =  sumjiebaROUGELr + rows[i][38]
                sumjiebaROUGELp =  sumjiebaROUGELp + rows[i][39]
                sumjiebaROUGELf =  sumjiebaROUGELf + rows[i][40]

    #            grs = 0
    #            with open(zidian, "r", encoding="utf-8") as zidianfile:
    #                reader = csv.reader(zidianfile)
    #                for row in reader:
    #                    if row[0] in infer_tokens_jieba:
    #                        grs = 1
    #                        break
    #            rows[i][41] = grs
    #            sumgrs = sumgrs + rows[i][41]
                grs = 0
                total_tokens = len(infer_tokens_jieba)
                with open(zidian, "r", encoding="utf-8") as zidianfile:
                    reader = csv.reader(zidianfile)
                    for row in reader:
                        if row[0] in infer_tokens_jieba:
                            grs += 1
                if total_tokens==0:
                    grs = 0
                else:
                    grs /= total_tokens
                rows[i][41] = grs
                sumgrs = sumgrs + rows[i][41]


    # 遍历所有的字符串并添加到集合中



    #        richness = len(unique_strings)
    #        print(richness)

            avebleu1 = sumbleu1 / picnum
            avebleu2 = sumbleu2 / picnum
            avebleu2s = sumbleu2s / picnum
            avebleu3 = sumbleu3 / picnum
            avebleu3s = sumbleu3s / picnum
            avebleu4 = sumbleu4 / picnum
            avebleu4s = sumbleu4s / picnum
            aveMETEOR = sumMETEOR / picnum
            aveROUGE1r = sumROUGE1r / picnum
            aveROUGE1p = sumROUGE1p / picnum
            aveROUGE1f = sumROUGE1f / picnum
            aveROUGE2r = sumROUGE2r / picnum
            aveROUGE2p = sumROUGE2p / picnum
            aveROUGE2f = sumROUGE2f / picnum
            aveROUGELr = sumROUGELr / picnum
            aveROUGELp = sumROUGELp / picnum
            aveROUGELf = sumROUGELf / picnum
            avejiebableu1 = sumjiebableu1 / picnum
            avejiebableu2 = sumjiebableu2 / picnum
            avejiebableu2s = sumjiebableu2s / picnum
            avejiebableu3 = sumjiebableu3 / picnum
            avejiebableu3s = sumjiebableu3s / picnum
            avejiebableu4 = sumjiebableu4 / picnum
            avejiebableu4s = sumjiebableu4s / picnum
            avejiebaMETEOR = sumjiebaMETEOR / picnum
            avejiebaROUGE1r = sumjiebaROUGE1r / picnum
            avejiebaROUGE1p = sumjiebaROUGE1p / picnum
            avejiebaROUGE1f = sumjiebaROUGE1f / picnum
            avejiebaROUGE2r = sumjiebaROUGE2r / picnum
            avejiebaROUGE2p = sumjiebaROUGE2p / picnum
            avejiebaROUGE2f = sumjiebaROUGE2f / picnum
            avejiebaROUGELr = sumjiebaROUGELr / picnum
            avejiebaROUGELp = sumjiebaROUGELp / picnum
            avejiebaROUGELf = sumjiebaROUGELf / picnum
      #      avegrs = richness * sumgrs / picnum / picnum
            avegrs =sumgrs / picnum
            print(avegrs)
            with open(os.path.join(models_root,model_name + 'geogptinfer_all_score.csv'), "w", newline="", encoding="utf-8") as outfile:
                csv_writer = csv.writer(outfile)
                csv_writer.writerows(rows)
            averows[e][1] = avebleu1
            averows[e][2] = avebleu2
            averows[e][3] = avebleu2s
            averows[e][4] = avebleu3
            averows[e][5] = avebleu3s
            averows[e][6] = avebleu4
            averows[e][7] = avebleu4s
            averows[e][8] = aveMETEOR
            averows[e][9] = aveROUGE1r                                                          
            averows[e][10] = aveROUGE1p
            averows[e][11] = aveROUGE1f
            averows[e][12] = aveROUGE2r                                                          
            averows[e][13] = aveROUGE2p
            averows[e][14] = aveROUGE2f
            averows[e][15] = aveROUGELr                                                          
            averows[e][16] = aveROUGELp
            averows[e][17] = aveROUGELf
            averows[e][18] = avejiebableu1
            averows[e][19] = avejiebableu2
            averows[e][20] = avejiebableu2s
            averows[e][21] = avejiebableu3
            averows[e][22] = avejiebableu3s
            averows[e][23] = avejiebableu4
            averows[e][24] = avejiebableu4s
            averows[e][25] = avejiebaMETEOR
            averows[e][26] = avejiebaROUGE1r
            averows[e][27] = avejiebaROUGE1p
            averows[e][28] = avejiebaROUGE1f
            averows[e][29] = avejiebaROUGE2r                                                          
            averows[e][30] = avejiebaROUGE2p
            averows[e][31] = avejiebaROUGE2f
            averows[e][32] = avejiebaROUGELr                                                          
            averows[e][33] = avejiebaROUGELp
            averows[e][34] = avejiebaROUGELf
            averows[e][35] = avegrs



    with open(os.path.join(models_root,'all_score_' + exp_name + '.csv'), "w", newline="", encoding="utf-8") as outavefile:
        csv_writer = csv.writer(outavefile)
        csv_writer.writerows(averows)

100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 51.76it/s]
Some weights of the model checkpoint at D:/code/geovit/models/checkpoint-7500 were not used when initializing ViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at D:/code/geogpt/gpt2_geo/checkpoint-128000 and are newly initialized: ['transformer.h.1.crossattention.masked_bias', 'transformer.h.5.crossattention.masked_bias', 'transformer.h.1.crossat

ok1

ok2



Map:   0%|          | 0/1173 [00:00<?, ? examples/s]

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

You are adding a <class 'transformers.integrations.TensorBoardCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
TensorBoardCallback
D:\software\anaconda\lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


...

100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 330.53it/s]
Some weights of the model checkpoint at D:/code/geovit/models/checkpoint-7500 were not used when initializing ViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at benjamin/gpt2-wechsel-chinese and are newly initialized: ['transformer.h.2.crossattention.c_proj.bias', 'transformer.h.3.crossattention.c_proj.weight', 'transformer.h.11.crossattention.q

ok1



Using pad_token, but it is not set yet.


ok2



ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.